# Multi-Camera Feature Extraction Example

This notebook demonstrates how to download and process 4 synchronized camera videos to extract ML-ready features.

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from download_and_extract_features import process_video_set, batch_process_timestamps

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Example 1: Process Single Video Set

Download 4 synchronized camera videos for one timestamp and extract features.

In [2]:
# Process single 15-minute video set
timestamp = "2025-10-20-06-00-45"

df = process_video_set(
    timestamp=timestamp,
    dataset='small',              # or 'full'
    download_dir='videos',
    output_dir='features_output',
    time_window_seconds=300,      # 5-minute windows
    skip_download=False,          # Set True to skip if already downloaded
    keep_videos=False             # Set True to keep videos after processing
)

2025-12-05 12:21:02 - download_and_extract_features - INFO - Pipeline initialized: small dataset
2025-12-05 12:21:02 - download_and_extract_features - INFO - Download dir: videos
2025-12-05 12:21:02 - download_and_extract_features - INFO - Output dir: features_output
2025-12-05 12:21:02 - download_and_extract_features - INFO - ================================================================================
2025-12-05 12:21:02 - download_and_extract_features - INFO - Processing video set: 2025-10-20-06-00-45
2025-12-05 12:21:02 - download_and_extract_features - INFO - ================================================================================
2025-12-05 12:21:02 - download_and_extract_features - INFO - Step 1: Finding videos in GCS bucket...
2025-12-05 12:21:02 - download_and_extract_features - INFO - Searching for videos with timestamp: 2025-10-20-06-00-45
2025-12-05 12:21:13 - download_and_extract_features - INFO - ✓ Found all 4 camera videos
2025-12-05 12:21:13 - download_and_ex

In [3]:
# Inspect the results
print(f"Shape: {df.shape}")
print(f"\nColumns: {len(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Shape: (1, 349)

Columns: 349

First few rows:


,window_idx,start_time,end_time,window_duration,hour_of_day,day_of_week,is_weekend,is_rush_hour,time_since_midnight,north_entry_count,...,EW_through_flow,speed_range,north_entry_congestion,north_exit_congestion,east_entry_congestion,east_exit_congestion,south_entry_congestion,south_exit_congestion,west_entry_congestion,west_exit_congestion
0,0,0.0,300.0,300.0,6,0,0,0,6.0,2,...,0.5,26.757543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Example 2: Batch Process Multiple Timestamps

Process multiple video sets and combine into training dataset.

In [ ]:
# List of timestamps to process
timestamps = [
    "2025-10-20-06-00-45",
    "2025-10-20-07-00-45",
    "2025-10-20-08-00-45",
]

# Batch process
results = batch_process_timestamps(
    timestamps=timestamps,
    dataset='small',
    skip_download=True  # Skip if already processed
)

print(f"Processed {len(results)}/{len(timestamps)} video sets")

In [ ]:
# Combine all into single DataFrame
all_features = []
for timestamp, df_subset in results.items():
    df_subset['video_timestamp'] = timestamp
    all_features.append(df_subset)

combined_df = pd.concat(all_features, ignore_index=True)
print(f"\nCombined dataset: {combined_df.shape}")
combined_df.head()

## Explore Features

In [ ]:
# Feature categories
temporal_cols = [col for col in df.columns if col in [
    'hour_of_day', 'day_of_week', 'is_weekend', 'is_rush_hour'
]]

camera_cols = {
    'north': [col for col in df.columns if col.startswith('north_')],
    'east': [col for col in df.columns if col.startswith('east_')],
    'south': [col for col in df.columns if col.startswith('south_')],
    'west': [col for col in df.columns if col.startswith('west_')],
}

roundabout_cols = [col for col in df.columns if col.startswith(('total_', 'avg_'))]
target_cols = [col for col in df.columns if col.endswith('_congestion')]

print("Feature Categories:")
print(f"  Temporal: {len(temporal_cols)}")
print(f"  North camera: {len(camera_cols['north'])}")
print(f"  East camera: {len(camera_cols['east'])}")
print(f"  South camera: {len(camera_cols['south'])}")
print(f"  West camera: {len(camera_cols['west'])}")
print(f"  Roundabout-wide: {len(roundabout_cols)}")
print(f"  Target columns: {len(target_cols)}")

## Visualize Key Metrics

In [ ]:
# Plot entry flows over time for each camera
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Entry Flow Rates by Camera', fontsize=16)

directions = ['north', 'east', 'south', 'west']
for idx, direction in enumerate(directions):
    ax = axes[idx // 2, idx % 2]
    col = f'{direction}_entry_flow_rate'
    
    if col in df.columns:
        ax.plot(df['start_time'] / 60, df[col], marker='o')
        ax.set_xlabel('Time (minutes)')
        ax.set_ylabel('Flow Rate (vehicles/min)')
        ax.set_title(f'{direction.capitalize()} Entrance')
        ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Compare entry flows across cameras
entry_cols = [f'{d}_entry_flow_rate' for d in directions]
if all(col in df.columns for col in entry_cols):
    plt.figure(figsize=(12, 6))
    
    for col, direction in zip(entry_cols, directions):
        plt.plot(df['start_time'] / 60, df[col], 
                marker='o', label=direction.capitalize())
    
    plt.xlabel('Time (minutes)')
    plt.ylabel('Entry Flow Rate (vehicles/min)')
    plt.title('Entry Flow Rates - All Cameras')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Roundabout-wide metrics
if 'total_circulating_occupancy' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Circulating occupancy
    axes[0].plot(df['start_time'] / 60, df['total_circulating_occupancy'], 
                marker='o', color='red')
    axes[0].set_xlabel('Time (minutes)')
    axes[0].set_ylabel('Vehicles')
    axes[0].set_title('Total Circulating Occupancy')
    axes[0].grid(True)
    
    # Average speed
    if 'avg_speed_km_h' in df.columns:
        axes[1].plot(df['start_time'] / 60, df['avg_speed_km_h'], 
                    marker='o', color='green')
        axes[1].set_xlabel('Time (minutes)')
        axes[1].set_ylabel('Speed (km/h)')
        axes[1].set_title('Average Speed Across Roundabout')
        axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

## Summary Statistics

In [ ]:
# Key metrics summary
key_metrics = [
    'total_entry_count', 'total_exit_count',
    'total_entry_flow_rate', 'total_exit_flow_rate',
    'total_circulating_occupancy', 'avg_speed_km_h'
]

available_metrics = [m for m in key_metrics if m in df.columns]
if available_metrics:
    print("Key Metrics Summary:")
    print(df[available_metrics].describe().round(2))

In [ ]:
# Camera comparison
print("\nAverage Entry Flow by Camera:")
for direction in directions:
    col = f'{direction}_entry_flow_rate'
    if col in df.columns:
        print(f"  {direction.capitalize():6s}: {df[col].mean():.2f} vehicles/min")

## Add Congestion Labels (Example)

This is where you would add your actual congestion labels for training.

In [ ]:
# Example: Simple rule-based labeling (replace with actual labels)
def label_congestion(flow_rate):
    """Example labeling function - replace with actual labels"""
    if pd.isna(flow_rate):
        return None
    elif flow_rate < 2:
        return 'free flowing'
    elif flow_rate < 4:
        return 'light delay'
    elif flow_rate < 6:
        return 'moderate delay'
    else:
        return 'heavy delay'

# Apply to all target columns
for direction in directions:
    entry_col = f'{direction}_entry_flow_rate'
    exit_col = f'{direction}_exit_flow_rate'
    
    if entry_col in df.columns:
        df[f'{direction}_entry_congestion'] = df[entry_col].apply(label_congestion)
    
    if exit_col in df.columns:
        df[f'{direction}_exit_congestion'] = df[exit_col].apply(label_congestion)

# Check distribution
print("Congestion Label Distribution (Example):")
print(df['north_entry_congestion'].value_counts())

## Save Processed Data

In [ ]:
# Save to CSV
output_file = f'features_output/labeled_features_{timestamp}.csv'
df.to_csv(output_file, index=False)
print(f"Saved to: {output_file}")

## Next Steps

1. **Collect more data**: Process multiple timestamps
2. **Add real labels**: Replace example labels with actual congestion annotations
3. **Feature selection**: Use feature importance to select best features
4. **Train model**: Use scikit-learn, XGBoost, or deep learning
5. **Evaluate**: Test on held-out time periods

### Model Training Example

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split

# Prepare features and targets
feature_cols = [col for col in df.columns if not col.endswith('_congestion')]
target_cols = [col for col in df.columns if col.endswith('_congestion')]

X = df[feature_cols]
y = df[target_cols]

# Encode labels
from sklearn.preprocessing import LabelEncoder
y_encoded = y.apply(LabelEncoder().fit_transform)

# Split data (preserve temporal order)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, shuffle=False
)

# Train model
model = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=100, random_state=42)
)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
```

In [3]:
# 1. Process multiple timestamps to create training dataset
from process_with_labels import batch_process_with_labels

timestamps = [
    "2025-10-20-06-00-45",
    "2025-10-20-07-00-45",
    "2025-10-20-08-00-45",
]
# timestamps = [
#     "2025-10-20-06-00-45"
# ]
df = batch_process_with_labels(
    timestamps=timestamps,
    output_combined='training_dataset.csv'
)

2025-12-05 12:43:10 - process_with_labels - INFO - Batch processing 3 timestamps...
2025-12-05 12:43:10 - process_with_labels - INFO - 
2025-12-05 12:43:10 - process_with_labels - INFO - Processing 1/3: 2025-10-20-06-00-45
2025-12-05 12:43:10 - process_with_labels - INFO - ================================================================================

2025-12-05 12:43:10 - process_with_labels - INFO - ================================================================================
2025-12-05 12:43:10 - process_with_labels - INFO - COMPLETE PIPELINE WITH LABELS: 2025-10-20-06-00-45
2025-12-05 12:43:10 - process_with_labels - INFO - ================================================================================
2025-12-05 12:43:10 - process_with_labels - INFO - 
Phase 1: Download videos and extract features...
2025-12-05 12:43:10 - process_with_labels - INFO - --------------------------------------------------------------------------------
2025-12-05 12:43:10 - download_and_extract_fe

In [5]:
df.columns.tolist()

['window_idx',
 'start_time',
 'end_time',
 'window_duration',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'is_rush_hour',
 'time_since_midnight',
 'north_entry_count',
 'north_exit_count',
 'north_entry_flow_rate',
 'north_exit_flow_rate',
 'north_entry_count_car',
 'north_entry_count_truck',
 'north_entry_count_bus',
 'north_exit_count_car',
 'north_exit_count_truck',
 'north_exit_count_bus',
 'north_circulating_occupancy_avg',
 'north_circulating_occupancy_max',
 'north_circulating_flow_rate',
 'north_speed_avg_km_h',
 'north_speed_std_km_h',
 'north_entry_speed_avg_km_h',
 'north_circulating_speed_avg_km_h',
 'north_entry_exit_balance',
 'north_entry_circulating_ratio',
 'north_circulating_density',
 'north_entry_count_lag_1',
 'north_entry_count_lag_2',
 'north_entry_count_lag_3',
 'north_entry_flow_rate_lag_1',
 'north_entry_flow_rate_lag_2',
 'north_entry_flow_rate_lag_3',
 'north_exit_count_lag_1',
 'north_exit_count_lag_2',
 'north_exit_count_lag_3',
 'north_exit_flow_rate

In [9]:
import pandas as pd
pd.read_csv('./dataset/Train.csv')

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16071,0YkHacyJMq-vnvgPjoXZqA0,Norman Niles #4,time_segment_4982_Norman Niles #4_congestion_e...,time_segment_4982_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-17-55-45.mp4,2025-10-26 17:55:45,2025-10-26 17:55:40,2025-10-26 17:56:40,2025-10-26,low,moderate delay,free flowing,4982,train
16072,PYoHaan6Lb-envgP6uGTmAE,Norman Niles #4,time_segment_4983_Norman Niles #4_congestion_e...,time_segment_4983_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-17-56-45.mp4,2025-10-26 17:56:45,2025-10-26 17:56:41,2025-10-26 17:57:59,2025-10-26,high,moderate delay,free flowing,4983,train
16073,7ooHafDLFdP_ld8Piv3HsQo,Norman Niles #4,time_segment_4984_Norman Niles #4_congestion_e...,time_segment_4984_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-17-57-45.mp4,2025-10-26 17:57:45,2025-10-26 17:57:42,2025-10-26 17:58:00,2025-10-26,medium,light delay,free flowing,4984,train
16074,0okHabnnG-K2nvgPsZGZ4Aw,Norman Niles #4,time_segment_4985_Norman Niles #4_congestion_e...,time_segment_4985_Norman Niles #4_congestion_e...,normanniles4/normanniles4_2025-10-26-17-58-45.mp4,2025-10-26 17:58:45,2025-10-26 17:58:42,2025-10-26 17:59:57,2025-10-26,none,light delay,free flowing,4985,train


What It Does (Step by Step)
1. Discovers 3 Timestamps from Train.csv
Reads dataset/Train.csv
Extracts unique video timestamps from the videos column (e.g., "2025-10-20-06-00-45")
Takes only the first 3 timestamps (due to limit=3)
Sorts them chronologically
2. Processes 3 Video Sets in Parallel with 4 Workers
For each of the 3 timestamps, it will: Per Timestamp (e.g., "2025-10-20-06-00-45"):
Download: 4 synchronized videos from GCS bucket
normanniles1/normanniles1_2025-10-20-06-00-45.mp4 (North camera)
normanniles2/normanniles2_2025-10-20-06-00-45.mp4 (East camera)
normanniles3/normanniles3_2025-10-20-06-00-45.mp4 (South camera)
normanniles4/normanniles4_2025-10-20-06-00-45.mp4 (West camera)
Track Objects: Run YOLO + ByteTrack on each video
Detect vehicles (car, motorcycle, bus, truck)
Track them through zones (entry, exit, circulating)
Calculate speeds, journey times
Extract Features: For each camera separately
Entry flow metrics (count, rate, queue length)
Circulating metrics (occupancy, density)
Exit flow metrics (count, OD matrix)
Temporal features (hour, day, rush_hour)
Lag features (t-1, t-2, t-3)
Rolling features (15min, 30min, 60min averages)
Merge Multi-Camera: Combine 4 camera features into one dataset
Camera-specific: north_entry_flow_rate, east_exit_count, etc.
Roundabout-wide: total_entry_count, avg_speed_km_h
Cross-camera: entry_flow_imbalance, NS_entry_balance
Directional: NS_total_entry, EW_through_flow
Add Training Labels: Match with Train.csv
Find 1-minute labeled videos within each 5-minute feature window
Aggregate labels (mode/max/last)
Add congestion labels: north_entry_congestion, north_exit_congestion, etc.
Add metadata: north_signaling, north_label_count
Save: features_output/labeled_features_2025-10-20-06-00-45.csv
Parallel Execution:
With workers=4, up to 4 timestamps can process simultaneously
Since limit=3, all 3 will run in parallel (assuming 4 workers available)
Progress bar shows: Processing: 3/3 [✓: 3, ✗: 0]
3. Combines Results
After all 3 timestamps finish:
Loads all 3 CSVs: labeled_features_*.csv
Concatenates them into one DataFrame
Adds video_timestamp column to track which timestamp each row came from
Saves combined result to training_dataset.csv
4. Returns DataFrame
The returned df contains: Structure:
df.shape
# (~54 rows, ~150 columns)
# 54 rows = 3 timestamps × ~18 windows per timestamp
# Each 15-minute video → ~3 windows of 5 minutes
Columns:
Temporal: start_time, end_time, hour_of_day, is_rush_hour, day_of_week
North camera: north_entry_flow_rate, north_exit_count, north_circulating_occupancy_avg, etc.
East camera: east_entry_flow_rate, east_exit_count, etc.
South camera: south_entry_flow_rate, south_exit_count, etc.
West camera: west_entry_flow_rate, west_exit_count, etc.
Roundabout-wide: total_entry_count, total_exit_count, avg_speed_km_h, etc.
Cross-camera: entry_flow_imbalance, NS_entry_balance, EW_entry_balance, etc.
Labels (8 columns):
north_entry_congestion, north_exit_congestion
east_entry_congestion, east_exit_congestion
south_entry_congestion, south_exit_congestion
west_entry_congestion, west_exit_congestion
Metadata: video_timestamp, window_idx, north_label_count, etc.
Example Output
df = parallel_process_all(workers=4, limit=3)

print(df.shape)
# (54, 152)

print(df['video_timestamp'].unique())
# ['2025-10-20-06-00-45', '2025-10-20-06-15-45', '2025-10-20-06-30-45']

print(df[['north_entry_congestion', 'north_exit_congestion']].head())
#   north_entry_congestion north_exit_congestion
# 0       free flowing           free flowing
# 1       free flowing           free flowing
# 2       light delay            free flowing
# ...

print(df[['total_entry_count', 'avg_speed_km_h', 'entry_flow_imbalance']].head())
#   total_entry_count  avg_speed_km_h  entry_flow_imbalance
# 0              42          28.5              0.15
# 1              38          31.2              0.08
# 2              51          25.3              0.22
# ...
Time Estimate
With 4 workers and 3 timestamps:
Download: ~1-2 min per timestamp (4 videos × 15min each)
Tracking: ~3-5 min per timestamp (4 videos × YOLO/ByteTrack)
Feature extraction: ~30 sec per timestamp
Label integration: ~10 sec per timestamp
Total: ~5-8 minutes (since 3 run in parallel with 4 workers)
Summary
In one line: Downloads 3 sets of 4-camera videos, tracks vehicles, extracts traffic features, adds congestion labels, and returns a ML-ready DataFrame with ~54 rows × ~150 columns.

In [10]:
from parallel_process_dataset import parallel_process_all

# Test with 10 timestamps
df = parallel_process_all(workers=4, limit=3)
print(f"Shape: {df.shape}")

# Process entire dataset
df_full = parallel_process_all(
    workers=4,
    output_combined='training_dataset.csv'
)

2025-12-05 12:50:28 - parallel_process_dataset - INFO - Loading dataset/Train.csv...
2025-12-05 12:50:28 - parallel_process_dataset - INFO - Loaded 16076 training records
2025-12-05 12:50:28 - parallel_process_dataset - INFO - Found 3 unique timestamps in Train.csv
2025-12-05 12:50:28 - parallel_process_dataset - INFO - Limiting to first 3 timestamps
2025-12-05 12:50:28 - parallel_process_dataset - INFO - Processing 3 timestamps with 4 workers...
Processing:   0%|          | 0/3 [00:00<?, ?it/s]2025-12-05 12:50:29 - process_with_labels - INFO - ================================================================================
2025-12-05 12:50:29 - process_with_labels - INFO - COMPLETE PIPELINE WITH LABELS: 2025-10-20-06-02-45
2025-12-05 12:50:29 - process_with_labels - INFO - ================================================================================
2025-12-05 12:50:29 - process_with_labels - INFO - 
Phase 1: Download videos and extract features...
2025-12-05 12:50:29 - process_wit

Shape: (3, 358)


Processing:   0%|          | 0/4019 [00:00<?, ?it/s]2025-12-05 12:53:33 - process_with_labels - INFO - ================================================================================
2025-12-05 12:53:33 - process_with_labels - INFO - COMPLETE PIPELINE WITH LABELS: 2025-10-20-06-00-45
2025-12-05 12:53:33 - process_with_labels - INFO - ================================================================================
2025-12-05 12:53:33 - process_with_labels - INFO - 
Phase 1: Download videos and extract features...
2025-12-05 12:53:33 - process_with_labels - INFO - --------------------------------------------------------------------------------
2025-12-05 12:53:33 - download_and_extract_features - INFO - Pipeline initialized: small dataset
2025-12-05 12:53:33 - download_and_extract_features - INFO - Download dir: videos
2025-12-05 12:53:33 - download_and_extract_features - INFO - Output dir: features_output
2025-12-05 12:53:33 - download_and_extract_features - INFO - ====================

KeyboardInterrupt: 

In [ ]:
from merge_with_train import merge_features_with_train

merged_df = merge_features_with_train(
    train_csv='dataset/Train.csv',
    features_csv='training_dataset.csv',
    output_csv='final_training_dataset.csv'
)

# All Train.csv columns preserved
print(merged_df[['responseId', 'view_label', 'congestion_enter_rating']].head())

# Features added with prefix
print(merged_df[['feature_north_entry_flow_rate', 'feature_total_entry_count']].head())